# **Imports**

In [ ]:
!pip install torch transformers scikit-learn --quiet

import os, re, json, time, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

In [ ]:
CFG = {
    "model_name":   "bert-base-uncased",
    "max_length":   256,
    "batch_size":   16,
    "epochs":       5,
    "lr":           2e-5,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "patience":     2,
    "seed":         42,
    "save_dir":     "./bert_newsgroups_6class",
    "device":       "cuda" if torch.cuda.is_available() else "cpu",
}
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
print(f"Device: {CFG['device']}")

Device: cuda


# **Data Loading**

In [ ]:
raw = fetch_20newsgroups(
    subset="all",
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=CFG["seed"],
)

label_consolidation = {
    # Religion
    "alt.atheism":              "religion",
    "soc.religion.christian":   "religion",
    "talk.religion.misc":       "religion",
    # Computers
    "comp.graphics":            "computers",
    "comp.os.ms-windows.misc":  "computers",
    "comp.sys.ibm.pc.hardware": "computers",
    "comp.sys.mac.hardware":    "computers",
    "comp.windows.x":           "computers",
    # Sports
    "rec.sport.baseball":       "sports",
    "rec.sport.hockey":         "sports",
    # Vehicles
    "rec.autos":                "vehicles",
    "rec.motorcycles":          "vehicles",
    # Science
    "sci.crypt":                "science",
    "sci.electronics":          "science",
    "sci.med":                  "science",
    "sci.space":                "science",
    # Politics
    "talk.politics.guns":       "politics",
    "talk.politics.mideast":    "politics",
    "talk.politics.misc":       "politics",
    # Marketplace
    "misc.forsale":             "marketplace and business",
}

original_class_names = raw.target_names
text_labels = [label_consolidation[original_class_names[l]] for l in raw.target]

le          = LabelEncoder()
labels      = le.fit_transform(text_labels)
class_names = le.classes_
num_labels  = len(class_names)
print(f"Classes ({num_labels}): {class_names.tolist()}")

counts = Counter(text_labels)
for c, n in sorted(counts.items()):
    print(f"  {c}: {n}")

Classes (7): ['computers', 'marketplace and business', 'politics', 'religion', 'science', 'sports', 'vehicles']
  computers: 4891
  marketplace and business: 975
  politics: 2625
  religion: 2424
  science: 3952
  sports: 1993
  vehicles: 1986


# **Model Pipeline**

### **Data Splits**

In [ ]:
texts = raw.data

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=CFG["seed"]
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, stratify=temp_labels, random_state=CFG["seed"]
)

def clean_text(text):
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

train_texts = [clean_text(t) for t in train_texts]
val_texts   = [clean_text(t) for t in val_texts]
test_texts  = [clean_text(t) for t in test_texts]
print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")

Train: 15076 | Val: 1885 | Test: 1885


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=CFG["max_length"],
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "token_type_ids": self.encodings["token_type_ids"][idx],
            "labels":         self.labels[idx],
        }

train_ds = NewsDataset(train_texts, train_labels)
val_ds   = NewsDataset(val_texts,   val_labels)
test_ds  = NewsDataset(test_texts,  test_labels)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],   shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"]*2, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"]*2, shuffle=False, num_workers=2, pin_memory=True)
print("Dataloaders ready.")

Dataloaders ready.


**Model Loading and Configuration**

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    CFG["model_name"],
    num_labels=num_labels,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    classifier_dropout=0.1,
)
model = model.to(CFG["device"])
print(f"Loaded {CFG['model_name']} | {num_labels} classes")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded bert-base-uncased | 7 classes


In [ ]:
total_steps  = len(train_loader) * CFG["epochs"]
warmup_steps = int(total_steps * CFG["warmup_ratio"])

no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_params = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": CFG["weight_decay"]},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],     "weight_decay": 0.0},
]
optimizer = AdamW(optimizer_grouped_params, lr=CFG["lr"], eps=1e-8)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
print(f"Total steps: {total_steps} | Warmup: {warmup_steps}")

Total steps: 4715 | Warmup: 471


**Training**

In [ ]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            ids   = batch["input_ids"].to(CFG["device"])
            mask  = batch["attention_mask"].to(CFG["device"])
            ttype = batch["token_type_ids"].to(CFG["device"])
            lbls  = batch["labels"].to(CFG["device"])
            out   = model(input_ids=ids, attention_mask=mask, token_type_ids=ttype, labels=lbls)
            loss  = out.loss
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
            correct    += (out.logits.argmax(-1) == lbls).sum().item()
            total      += lbls.size(0)
            total_loss += loss.item() * lbls.size(0)
    return total_loss / total, correct / total

best_val_loss, best_weights, patience_count = float("inf"), None, 0
history = []

for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    print(f"Epoch {epoch}/{CFG['epochs']} | train_loss={tr_loss:.4f} acc={tr_acc:.4f} | "
          f"val_loss={vl_loss:.4f} acc={vl_acc:.4f} | {time.time()-t0:.0f}s")

    history.append(dict(epoch=epoch, tr_loss=tr_loss, tr_acc=tr_acc,
                        vl_loss=vl_loss, vl_acc=vl_acc))

    if vl_loss < best_val_loss:
        best_val_loss  = vl_loss
        best_weights   = copy.deepcopy(model.state_dict())
        patience_count = 0
        print("  ✓ Best model updated")
    else:
        patience_count += 1
        print(f"  No improvement ({patience_count}/{CFG['patience']})")
        if patience_count >= CFG["patience"]:
            print("  Early stopping triggered.")
            break

model.load_state_dict(best_weights)
print("Training complete.")

Epoch 1/5 | train_loss=0.8673 acc=0.7022 | val_loss=0.4761 acc=0.8451 | 679s
  ✓ Best model updated
Epoch 2/5 | train_loss=0.3630 acc=0.8825 | val_loss=0.4504 acc=0.8578 | 680s
  ✓ Best model updated
Epoch 3/5 | train_loss=0.2257 acc=0.9302 | val_loss=0.5338 acc=0.8546 | 680s
  No improvement (1/2)
Epoch 4/5 | train_loss=0.1470 acc=0.9534 | val_loss=0.6004 acc=0.8631 | 680s
  No improvement (2/2)
  Early stopping triggered.
Training complete.


In [ ]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for batch in test_loader:
        ids   = batch["input_ids"].to(CFG["device"])
        mask  = batch["attention_mask"].to(CFG["device"])
        ttype = batch["token_type_ids"].to(CFG["device"])
        lbls  = batch["labels"].to(CFG["device"])
        logits = model(input_ids=ids, attention_mask=mask, token_type_ids=ttype).logits
        all_preds.extend(logits.argmax(-1).cpu().numpy())
        all_true.extend(lbls.cpu().numpy())

print(f"Test Accuracy: {accuracy_score(all_true, all_preds):.4f}\n")
print(classification_report(all_true, all_preds, target_names=class_names, digits=3))

Test Accuracy: 0.8668

                          precision    recall  f1-score   support

               computers      0.855     0.926     0.889       489
marketplace and business      0.885     0.794     0.837        97
                politics      0.828     0.847     0.838       262
                religion      0.914     0.835     0.873       243
                 science      0.865     0.823     0.843       396
                  sports      0.967     0.890     0.927       200
                vehicles      0.806     0.884     0.843       198

                accuracy                          0.867      1885
               macro avg      0.874     0.857     0.864      1885
            weighted avg      0.869     0.867     0.867      1885



In [ ]:
os.makedirs(CFG["save_dir"], exist_ok=True)

model.save_pretrained(CFG["save_dir"])
tokenizer.save_pretrained(CFG["save_dir"])

# Save label map
label_map = {i: name for i, name in enumerate(class_names)}
with open(f"{CFG['save_dir']}/label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

# Save .pth
torch.save({
    "model_state_dict":     model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "best_val_loss":        best_val_loss,
    "history":              history,
    "config":               CFG,
    "num_labels":           num_labels,
    "class_names":          class_names.tolist(),
    "label_consolidation":  label_consolidation,  # save mapping for deployment
}, f"{CFG['save_dir']}/bert_6class.pth")

print(f"Saved to {CFG['save_dir']}/")

from google.colab import files
files.download(f"{CFG['save_dir']}/bert_6class.pth")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ./bert_newsgroups_6class/


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Testing**

In [ ]:
class NewsgroupsClassifier:
    def __init__(self, model_dir: str, device: str = None):
        self.device    = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model     = AutoModelForSequenceClassification.from_pretrained(model_dir).to(self.device)
        self.model.eval()
        with open(f"{model_dir}/label_map.json") as f:
            self.label_map = {int(k): v for k, v in json.load(f).items()}

    def _clean(self, text: str) -> str:
        text = re.sub(r"[^\x00-\x7F]+", " ", text)
        return re.sub(r"\s+", " ", text).strip()

    def predict(self, text: str, top_k: int = 3) -> dict:
        enc = self.tokenizer(
            self._clean(text), truncation=True,
            max_length=256, padding="max_length", return_tensors="pt"
        ).to(self.device)
        with torch.no_grad():
            probs = torch.softmax(self.model(**enc).logits, dim=-1)[0].cpu().numpy()
        top = probs.argsort()[::-1][:top_k]
        return {
            "label":      self.label_map[int(top[0])],
            "confidence": float(probs[top[0]]),
            "top_k":      [{"label": self.label_map[int(i)], "prob": float(probs[i])} for i in top],
        }

    def predict_batch(self, texts: list, batch_size: int = 32) -> list:
        results = []
        for i in range(0, len(texts), batch_size):
            batch = [self._clean(t) for t in texts[i: i+batch_size]]
            enc   = self.tokenizer(batch, truncation=True, max_length=256,
                                   padding=True, return_tensors="pt").to(self.device)
            with torch.no_grad():
                probs = torch.softmax(self.model(**enc).logits, dim=-1).cpu().numpy()
            for p in probs:
                top_i = p.argmax()
                results.append({"label": self.label_map[int(top_i)], "confidence": float(p[top_i])})
        return results

# Smoke test
clf = NewsgroupsClassifier(CFG["save_dir"])
tests = [
    "NASA launched a new Mars rover today.",
    "My GPU temperature keeps hitting 95 degrees.",
    "The bishop gave a sermon on forgiveness.",
    "Sale: IBM Thinkpad, excellent condition, $250.",
    "The Canadiens beat the Bruins 4-2 in overtime.",
    "The senator proposed a new gun control bill.",
]
for text in tests:
    r = clf.predict(text)
    print(f"  {r['label']:12s} ({r['confidence']*100:.1f}%)  →  {text[:55]}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  science      (98.8%)  →  NASA launched a new Mars rover today.
  computers    (87.6%)  →  My GPU temperature keeps hitting 95 degrees.
  religion     (99.1%)  →  The bishop gave a sermon on forgiveness.
  marketplace and business (96.6%)  →  Sale: IBM Thinkpad, excellent condition, $250.
  sports       (99.4%)  →  The Canadiens beat the Bruins 4-2 in overtime.
  politics     (98.5%)  →  The senator proposed a new gun control bill.
